In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

driver = webdriver.Chrome()  # требуется chromedriver
driver.get("https://getmentor.dev/")

while True:
    try:
        # Сначала скроллим вниз, чтобы кнопка стала видна
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Ждём появления кнопки «Посмотреть ещё»
        button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.css-…"))  # замените на актуальный селектор
        )
        button.click()
        # Ждём подгрузки новых карточек
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".mentor-card-class"))  # замените
        )
    except Exception:
        break  # Если кнопка не найдена — выходим

# После загрузки всего контента
soup = BeautifulSoup(driver.page_source, "html.parser")


driver.quit()
   

In [21]:
import json
import pandas as pd
script_tag = soup.find('script', {'id': '__NEXT_DATA__'})
if script_tag:
    try:
        # Парсим JSON
        data = json.loads(script_tag.string)
        
        # Извлекаем список менторов
        mentors = data['props']['pageProps']['pageMentors']
        
        # Создаем DataFrame
        df = pd.DataFrame(mentors)
        
        # Переименовываем столбцы для удобства (опционально)
        column_mapping = {
            'airtableId': 'airtable_id',
            'photo_url': 'photo_url',
            'menteeCount': 'mentee_count'
        }
        df = df.rename(columns=column_mapping)
        
        # Преобразуем компетенции в список (опционально)
        if 'competencies' in df.columns:
            df['competencies'] = df['competencies'].apply(
                lambda x: [c.strip() for c in x.split(',')] if isinstance(x, str) else []
            )
        
        # Преобразуем числовые поля (опционально)
        numeric_cols = ['id', 'mentee_count']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Показываем результат
        print("DataFrame создан успешно!")
        print(f"Количество менторов: {len(df)}")
        print("\nПервые записи:")
        print(df.head())
        
        # Сохраняем в CSV (опционально)
        df.to_csv('mentors.csv', index=False, encoding='utf-8')
        print("\nДанные сохранены в 'mentors.csv'")
        
        # Сохраняем в Excel (опционально)
        # df.to_excel('mentors.xlsx', index=False)
        
    except json.JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
    except KeyError as e:
        print(f"Ключ не найден в JSON: {e}")
else:
    print("Тег скрипта не найден")

DataFrame создан успешно!
Количество менторов: 3024

Первые записи:
     id        airtable_id                    slug               name  \
0  2798  reckMOKgyc6upyHwL     nikolay-sheyko-2798     Nikolay Sheyko   
1  4991  recmLN5ebN5ondUnA      elena-galkina-4991      Елена Галкина   
2  3192  rectT0buN1Xdrj9ah  andrey-protopopov-3192  Андрей Протопопов   
3  3189  recgQ2QXCnM0haCBV   daniil-dzhumaylo-3189    Даниил Джумайло   
4  2754  recZsbR7xtww1Q9vh    zhembe-mihaylov-2754     Жэмбэ Михайлов   

                job         workplace  \
0               CTO       AI и грабли   
1              СЕО   "WorldLegalTeam"   
2     Founder / CEO           ReenCar   
3  Продакт-менеджер  Московская Биржа   
4     Founder | CEO   Friendly Family   

                                        competencies experience  \
0             [AI, LLM, ML, RAG, GPT, Google Gemini]       5-10   
1                           [Law, Legal, Career, HR]        10+   
2  [Leadership, management, product managemen

In [22]:
df.head()

,id,airtable_id,slug,name,job,workplace,competencies,experience,price,mentee_count,photo_url,tags,sortOrder,isVisible,sponsors,calendarType,isNew
0,2798,reckMOKgyc6upyHwL,nikolay-sheyko-2798,Nikolay Sheyko,CTO,AI и грабли,"[AI, LLM, ML, RAG, GPT, Google Gemini]",5-10,15000 руб,9,https://dl.airtable.com/.directUploadAttachmen...,"[Data Science/ML, Product Management, Team Lea...",1,True,none,none,False
1,4991,recmLN5ebN5ondUnA,elena-galkina-4991,Елена Галкина,СЕО,"""WorldLegalTeam""","[Law, Legal, Career, HR]",10+,Бесплатно,0,https://dl.airtable.com/.directUploadAttachmen...,"[Карьера, Собеседования, Аналитика, HR]",2,True,none,none,False
2,3192,rectT0buN1Xdrj9ah,andrey-protopopov-3192,Андрей Протопопов,Founder / CEO,ReenCar,"[Leadership, management, product management, d...",10+,По договоренности,2,https://dl.airtable.com/.attachments/2a6abef7c...,"[Entrepreneurship, Marketing, Team Lead/Manage...",3,True,none,calendly,False
3,3189,recgQ2QXCnM0haCBV,daniil-dzhumaylo-3189,Даниил Джумайло,Продакт-менеджер,Московская Биржа,"[Agile: Scrum, Kanban, Project Management, Pro...",10+,4000 руб,12,https://dl.airtable.com/.attachments/ae948189e...,"[Product Management, Project Management, Собес...",3,True,none,none,False
4,2754,recZsbR7xtww1Q9vh,zhembe-mihaylov-2754,Жэмбэ Михайлов,Founder | CEO,Friendly Family,[Проведение CustDev Создание прототипа продукт...,2-5,По договоренности,2,https://dl.airtable.com/.attachments/810234f07...,[Product Management],3,True,none,calendly,False
